# Zoom動画 自動編集（Google Colab版）

Zoomの録画を、**OP/EDカット → 登壇者の開始点で開始 → 無音カット → 登壇者以外の発話カット →（任意）AIで「もたつき・トラブル」区間を検出してカット → OP/ED素材をディゾルブで接続** まで自動で編集し、mp4を書き出すノートブックです。
書き出しには Colab の **無料GPU** が使えます（使える場合は NVENC で高速に処理します）。

コード本体は GitHub リポジトリ（KamikiKei/zoom-video-auto-editor）から毎回取得するので、**このノートブック1枚を配れば、コードは常に最新**になります。

## 準備するもの（Google Drive に置いてください）

| もの | 備考 |
|---|---|
| 録画ファイル（.mp4） | Zoomの録画本体 |
| Zoom文字起こし（.vtt） | Zoomのクラウド録画の「音声文字起こし」。**話者カットとAI検出に必要**です。無い場合は開始点の自動推定・話者カット・AI検出が働きません |
| OP/ED素材 | Canva等で作った素材。画像（png/jpg）または動画（mp4等）。頭とお尻に同じ素材を使います |

おすすめの置き方： マイドライブに `video-auto-editor` フォルダを作り、その中に `input`（材料）と `output`（成果物）のフォルダを作る。

素材のファイル名は自由です（リネーム不要）。UIのプルダウンから選ぶので、決まった名前に直す必要はありません。

## 実行順序

上のセルから順番に実行してください（Shift+Enter で次のセルへ）。

1. 環境確認（GPU / ffmpeg）
2. コード取得（GitHubから clone / 更新）
3. Google Drive マウント
4. 設定（Gemini APIキーの読み込み。ファイルの場所の指定は、UI方式では不要）
5. UIで編集（おすすめ）：プルダウンで素材を選んで、ボタン2つ（①カット案 → ②書き出し）で完了
6. （手動方式）plan → render → 結果確認：設定セルを書き換えて細かく指定したい場合は、下のセル群をどうぞ

## 注意

- **GPUランタイム**にしてください： メニュー「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータで **GPU（T4など）** を選択。無料枠のGPUは割り当て状況次第で、使えない時間帯もあります（その場合はCPU書き出しになりますが処理はできます。少し待ちます）。
- **セッションが切れても、出力ファイルは Drive に残ります**（出力先を Drive にしているため）。途中で切れた場合は、もう一度上から実行し直してください。plan（カット案）も Drive の出力フォルダに保存されるので、render だけ再実行することもできます。
- Gemini APIキー（AI検出用）は、ノートブックに直接書かず **Colab の Secrets** に登録してください（手順は設定セルのコメント参照）。キーが無い場合は AI検出なしでそのまま進みます。

## 1. 環境確認

GPUの有無と、ffmpeg で `h264_nvenc`（GPUエンコード）が使えるかを確認します。

In [ ]:
# GPUの確認（GPUランタイムでない場合は何も表示されません）
!nvidia-smi

# ffmpeg の確認
!ffmpeg -version | head -2

# NVENC が実際に使えるか、小さい映像で試しエンコードして確認
import subprocess
r = subprocess.run(
    ["ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc=size=256x256:rate=30:duration=0.5",
     "-c:v", "h264_nvenc", "-f", "null", "-"],
    capture_output=True, text=True)
print("h264_nvenc:", "使えます（GPUで高速に書き出します）" if r.returncode == 0
      else "使えません（CPUの libx264 で書き出します）")

## 2. コード取得

GitHub リポジトリを取得します（2回目以降は最新に更新します）。
このツール（plan / render）は Python 標準ライブラリだけで動くので、追加のインストールは不要です。

In [ ]:
%cd /content
!if [ -d zoom-video-auto-editor ]; then cd zoom-video-auto-editor && git pull; else git clone https://github.com/KamikiKei/zoom-video-auto-editor.git; fi
%cd /content/zoom-video-auto-editor

## 3. Google Drive マウント

許可ダイアログが出たら、Drive へのアクセスを許可してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 設定

材料ファイルの場所と、出力先を指定します。**自分の Drive の構成に合わせて書き換えてください。**
UIで編集する場合は、このセルのファイル指定を書き換える必要はありません（Gemini APIキーの読み込みのために、そのまま実行してください）。
ついでに、AI検出用の Gemini APIキーを Colab の Secrets から読み込みます。

### Gemini APIキーの登録方法（AI検出を使いたい人だけ）

1. 左端の鍵アイコン（Secrets）を開く
2. 「新しいシークレットを追加」で、名前を `GEMINI_API_KEY` にして、値にAPIキーを貼る
3. 「ノートブックからのアクセス」を ON にする

キーが登録されていない場合は、AI検出なしでそのまま進みます（話者カットや無音カットは動きます）。

In [ ]:
import os

# ===== ここを自分の環境に合わせて書き換える =====
BASE = "/content/drive/MyDrive/video-auto-editor"
INPUT_VIDEO   = f"{BASE}/input/recording.mp4"   # 録画 mp4
TRANSCRIPT_VTT = f"{BASE}/input/recording.vtt"  # Zoom文字起こし VTT（無い場合は "" にする）
ASSET_FILE    = f"{BASE}/input/op_ed.mp4"       # OP/ED素材（動画 or 画像 png/jpg）
OUTPUT_DIR    = f"{BASE}/output"                # 出力先フォルダ（Drive上）
OUTPUT_NAME   = "edited.mp4"                    # 出力ファイル名

SILENCE_CUT   = True   # 無音カットをするか
SILENCE_SEC   = 3.0    # 何秒以上の無音をカットするか
PRESENTERS    = ""     # 登壇者の話者番号（例 "0" や "0,1"）。空欄なら自動推定
ENCODER       = "auto" # auto / x264 / nvenc（autoならNVENCが使える環境ではNVENC）
# ===============================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
PLAN_PATH = f"{OUTPUT_DIR}/plan.json"
OUTPUT_PATH = f"{OUTPUT_DIR}/{OUTPUT_NAME}"

for label, p in [("動画", INPUT_VIDEO), ("素材", ASSET_FILE)]:
    print(("OK  " if os.path.exists(p) else "無い ") + f"{label}: {p}")
if TRANSCRIPT_VTT:
    print(("OK  " if os.path.exists(TRANSCRIPT_VTT) else "無い ")
          + f"文字起こし: {TRANSCRIPT_VTT}（無いと話者カット・AI検出が働きません）")

# Gemini APIキー（Colab Secrets から。キーの値をこのノートブックに書かないこと）
try:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
    if _key:
        os.environ["GEMINI_API_KEY"] = _key
        print("Gemini APIキー: 読み込みました（AI検出を使います）")
    else:
        print("Gemini APIキー: 未設定のため、AI検出なしで進みます")
except Exception:
    print("Gemini APIキー: 未設定のため、AI検出なしで進みます")

# AI検出のモデル（GEMINI_MODEL が未設定なら既定モデルを使う。変えたい場合は環境変数で指定）
os.environ.setdefault("GEMINI_MODEL", "gemini-3-flash-preview")
print(f"AI検出モデル: {os.environ['GEMINI_MODEL']}")


## 5. UIで編集（おすすめ）

設定セルを書き換えずに、プルダウンとボタンだけで編集できます。

### 使い方

1. 素材を Drive の `video-auto-editor/input` フォルダに置く（**ファイル名は自由。リネーム不要**）
2. 下のUIセルを実行して「素材フォルダを読み込む」を押し、動画・文字起こし・OP/ED素材をプルダウンで選ぶ
- 動画を選ぶと、同名の文字起こし（.vtt）があれば自動で選ばれます
- OP/ED素材が1つだけある場合は、最初から選ばれた状態になります
3. 「① カット案を作る」を押す → カット案の要約と話者一覧が出ます
4. 登壇者（残す話者）のチェックを確認する（自動推定が初期値。違えば直してください）
5. 「② 書き出す」を押す → 登壇者の選択を変えていたら、カット案を作り直してから書き出します。出力は Drive の `video-auto-editor/output` フォルダです

文字起こしが無いと、話者カット・AI検出・開始点の自動推定は働きません（無音カットとOP/ED接続はできます）。

In [ ]:
# ===== UIで編集（プルダウン＋ボタン） =====
# 設定セルを書き換えずに、素材の選択と実行をここで行います。
import glob
import json
import os
import re
import subprocess
import sys
import time
from collections import Counter

import ipywidgets as widgets
from IPython.display import display

UI_BASE = "/content/drive/MyDrive/video-auto-editor"
UI_INPUT_DIR = f"{UI_BASE}/input"
UI_OUTPUT_DIR = f"{UI_BASE}/output"
UI_PLAN_PATH = f"{UI_OUTPUT_DIR}/plan.json"
UI_SILENCE_SEC = 3.0  # 無音カットの最小秒数

os.makedirs(UI_OUTPUT_DIR, exist_ok=True)

# Gemini APIキー: 設定セルで読み込み済みのはず。無ければ Secrets から読む保険（値は表示しない）
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        _k = userdata.get("GEMINI_API_KEY")
        if _k:
            os.environ["GEMINI_API_KEY"] = _k
            print("Gemini APIキー: Secrets から読み込みました")
    except Exception:
        pass
os.environ.setdefault("GEMINI_MODEL", "gemini-3-flash-preview")


def mmss(sec):
    sec = int(round(sec))
    return f"{sec // 60}:{sec % 60:02d}"


_dd_layout = widgets.Layout(width="92%")
_dd_style = {"description_width": "110px"}

video_dd = widgets.Dropdown(description="動画", options=[],
                            layout=_dd_layout, style=_dd_style)
transcript_dd = widgets.Dropdown(description="文字起こし", options=[("（なし）", "")],
                                 layout=_dd_layout, style=_dd_style)
asset_dd = widgets.Dropdown(description="OP/ED素材", options=[],
                            layout=_dd_layout, style=_dd_style)
silence_check = widgets.Checkbox(value=True, description="無音カット", indent=False)
ai_check = widgets.Checkbox(value=True, description="AI検出", indent=False)
encoder_dd = widgets.Dropdown(description="エンコーダ", options=["auto", "nvenc", "x264"],
                              value="auto", style=_dd_style)
output_name_text = widgets.Text(description="出力ファイル名", value="edited.mp4",
                                style=_dd_style)

scan_btn = widgets.Button(description="素材フォルダを読み込む",
                          button_style="primary", icon="folder-open")
plan_btn = widgets.Button(description="① カット案を作る", button_style="primary")
render_btn = widgets.Button(description="② 書き出す", button_style="success")
plan_out = widgets.Output()
render_out = widgets.Output()
presenter_box = widgets.VBox(
    [widgets.HTML("「① カット案を作る」を押すと、ここに話者一覧（登壇者の確認用）が出ます")])

_state = {"plan_presenters": None}  # 直近の plan で使われた登壇者（render時の比較用）
presenter_checks = {}  # 話者番号 -> Checkbox


def _list_files(exts):
    return [p for p in sorted(glob.glob(os.path.join(UI_INPUT_DIR, "*")))
            if os.path.isfile(p) and p.lower().endswith(exts)]


def _match_transcript():
    # 選択中の動画と同名（拡張子だけ違う）の .vtt があれば、文字起こしをそれに合わせる。
    # 一致が無ければ「（なし）」のまま。
    v = video_dd.value
    if not v:
        return
    stem = os.path.splitext(v)[0].lower()
    for _label, val in transcript_dd.options:
        if val and os.path.splitext(val)[0].lower() == stem:
            transcript_dd.value = val
            return
    transcript_dd.value = ""


def _on_video_change(change):
    if change.get("name") == "value":
        _match_transcript()


video_dd.observe(_on_video_change, names="value")


def scan_inputs(_=None):
    videos = _list_files((".mp4", ".mov", ".mkv", ".webm"))
    vtts = _list_files((".vtt",))
    assets = _list_files((".mp4", ".mov", ".png", ".jpg", ".jpeg", ".webp"))
    video_dd.options = [(os.path.basename(p), p) for p in videos]
    transcript_dd.options = [("（なし）", "")] + [(os.path.basename(p), p) for p in vtts]
    asset_dd.options = [(os.path.basename(p), p) for p in assets]
    # options を入れ替えただけでは選択値が決まらないので、初期選択を明示する
    if videos and video_dd.value not in videos:
        video_dd.value = videos[0]
    if len(assets) == 1:
        asset_dd.value = assets[0]  # 素材が1つだけなら初期選択にする
    elif assets and asset_dd.value not in assets:
        asset_dd.value = assets[0]
    _match_transcript()
    print(f"素材フォルダ: {UI_INPUT_DIR}")
    print(f"動画 {len(videos)}件 / 文字起こし {len(vtts)}件 / OP・ED素材 {len(assets)}件")
    if not videos:
        print("動画が見つかりません。Drive の input フォルダに置いてから、もう一度読み込んでください。")


def _run_plan(presenters=None):
    # plan を実行して plan(dict) を返す。失敗したらメッセージを表示して None を返す
    video = video_dd.value
    if not video:
        print("動画が選ばれていません。「素材フォルダを読み込む」を押してください。")
        return None
    cmd = [sys.executable, "edit.py", "plan",
           "--video", video,
           "--out", UI_PLAN_PATH]
    if asset_dd.value:
        cmd += ["--thumb", asset_dd.value]
    vtt = transcript_dd.value
    if vtt:
        cmd += ["--transcript", vtt]
        if ai_check.value:
            cmd += ["--llm-detect"]  # キーが無ければ edit.py 側で自動スキップされる
    elif ai_check.value:
        print("（AI検出は文字起こしがある場合のみ働きます）")
    if silence_check.value:
        cmd += ["--silence-cut", "--silence-sec", str(UI_SILENCE_SEC)]
    if presenters:
        cmd += ["--presenters", ",".join(str(i) for i in presenters)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"plan の作成に失敗しました（returncode={r.returncode}）")
        print(r.stdout[-2000:])
        print(r.stderr[-3000:])
        return None
    plan = json.load(open(UI_PLAN_PATH, encoding="utf-8"))
    _state["plan_presenters"] = list(plan.get("presenters") or [])
    return plan


def _print_plan_summary(plan):
    cands = plan["silence"]["candidates"]
    print("=" * 50)
    print(f"元の尺　　　: {mmss(plan['video']['duration'])}")
    print(f"開始点　　　: {mmss(plan['trim_start'])}（{plan['trim_start_source']}）")
    print(f"終了点　　　: {mmss(plan['end'])}")
    print(f"想定の出力尺: {mmss(plan['expected_duration'])}")
    print(f"登壇者　　　: {plan['presenters']}（保持ポリシー適用: {plan['keep_policy']['applied']}）")
    by_src = Counter(c["source"] for c in cands if c["enabled"])
    label = {"silence": "無音", "ops": "運営の発話", "other": "登壇者以外の発話",
             "llm": "AI検出", "user": "手動"}
    print("カット候補（有効なもの）:")
    for src, n in by_src.items():
        print(f"  {label.get(src, src)}: {n}件")
    llm = plan.get("llm_detection") or {}
    print(f"AI検出　　　: {llm.get('status')} {llm.get('detail', '')}")
    print(f"plan保存先　: {UI_PLAN_PATH}")


def _build_presenter_checks(plan):
    # plan の話者一覧から登壇者チェックボックス群を作る（既定は plan の推定）
    presenter_checks.clear()
    sp = plan.get("speakers") or {}
    names = sp.get("names") or {}
    secs = sp.get("speakers") or {}
    chosen = set(plan.get("presenters") or [])
    idxs = sorted({int(k) for k in list(names.keys()) + list(secs.keys())})
    boxes = []
    for i in idxs:
        desc = f"話者{i}: {names.get(str(i), '名前なし')}"
        if str(i) in secs:
            desc += f"（発話 {mmss(secs[str(i)])}）"
        cb = widgets.Checkbox(value=(i in chosen), description=desc, indent=False)
        presenter_checks[i] = cb
        boxes.append(cb)
    if boxes:
        presenter_box.children = [widgets.HTML("<b>登壇者（残す話者）を確認してください</b>")] + boxes
    else:
        presenter_box.children = [widgets.HTML("話者情報がありません（文字起こしが無いと話者カットは働きません）")]


def on_plan_clicked(_):
    plan_out.clear_output()
    with plan_out:
        plan = _run_plan()
        if plan is None:
            return
        _print_plan_summary(plan)
        _build_presenter_checks(plan)


def on_render_clicked(_):
    render_out.clear_output()
    with render_out:
        if _state["plan_presenters"] is None or not os.path.exists(UI_PLAN_PATH):
            print("先に「① カット案を作る」を押してください。")
            return
        selected = sorted(i for i, cb in presenter_checks.items() if cb.value)
        if presenter_checks and not selected:
            print("登壇者が1人も選ばれていません。最低1人にチェックしてください。")
            return
        if selected != sorted(_state["plan_presenters"]):
            print(f"登壇者の選択が変わったため、カット案を作り直します（登壇者: {selected}）")
            plan = _run_plan(presenters=selected)
            if plan is None:
                return
            _print_plan_summary(plan)
            _build_presenter_checks(plan)
        out_name = output_name_text.value.strip() or "edited.mp4"
        out_path = os.path.join(UI_OUTPUT_DIR, out_name)
        cmd = [sys.executable, "edit.py", "render",
               "--plan", UI_PLAN_PATH,
               "-o", out_path,
               "--encoder", encoder_dd.value]
        if asset_dd.value:
            cmd += ["--thumb", asset_dd.value]
        t0 = time.time()
        r = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - t0
        if r.returncode != 0:
            print(f"書き出しに失敗しました（returncode={r.returncode}）")
            print(r.stdout[-2000:])
            print(r.stderr[-3000:])
            return
        print(r.stdout[-3000:])
        m = re.search(r"エンコーダ: (\S+)", r.stdout)
        print("=" * 50)
        print(f"使われたエンコーダ: {m.group(1) if m else '不明（上のログを確認）'}")
        print(f"所要時間　　　　　: {elapsed / 60:.1f} 分")
        print(f"出力　　　　　　　: {out_path}")


scan_btn.on_click(scan_inputs)
plan_btn.on_click(on_plan_clicked)
render_btn.on_click(on_render_clicked)

display(widgets.VBox([
    widgets.HTML(f"<b>素材フォルダ</b>: {UI_INPUT_DIR}"),
    scan_btn,
    video_dd,
    transcript_dd,
    asset_dd,
    widgets.HBox([silence_check, ai_check]),
    widgets.HBox([encoder_dd, output_name_text]),
    plan_btn,
    plan_out,
    presenter_box,
    render_btn,
    render_out,
]))
scan_inputs()

## 6. plan（カット案づくり）【手動方式】

UIで編集する場合は、この節と次の節は不要です。

カット案を作ります（まだ書き出しはしません）。
開始点・カット候補（無音／運営など登壇者以外の発話／AI検出）・想定の出力尺が表示されます。
おかしいところがあれば、設定セルに戻って直してから実行し直してください。

In [ ]:
import json, subprocess, sys

cmd = [sys.executable, "edit.py", "plan",
       "--video", INPUT_VIDEO,
       "--thumb", ASSET_FILE,
       "--out", PLAN_PATH]
if TRANSCRIPT_VTT:
    cmd += ["--transcript", TRANSCRIPT_VTT]
    cmd += ["--llm-detect"]  # キーが無ければ自動でスキップされます
if SILENCE_CUT:
    cmd += ["--silence-cut", "--silence-sec", str(SILENCE_SEC)]
if PRESENTERS:
    cmd += ["--presenters", PRESENTERS]

r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("plan の作成に失敗しました")

plan = json.load(open(PLAN_PATH, encoding="utf-8"))

def mmss(sec):
    sec = int(round(sec)); return f"{sec // 60}:{sec % 60:02d}"

cands = plan["silence"]["candidates"]
print("=" * 50)
print(f"元の尺　　　: {mmss(plan['video']['duration'])}")
print(f"開始点　　　: {mmss(plan['trim_start'])}（{plan['trim_start_source']}）")
print(f"終了点　　　: {mmss(plan['end'])}")
print(f"想定の出力尺: {mmss(plan['expected_duration'])}")
print(f"登壇者　　　: {plan['presenters']}（保持ポリシー適用: {plan['keep_policy']['applied']}）")
from collections import Counter
by_src = Counter(c["source"] for c in cands if c["enabled"])
label = {"silence": "無音", "ops": "運営の発話", "other": "登壇者以外の発話",
         "llm": "AI検出", "user": "手動"}
print("カット候補（有効なもの）:")
for src, n in by_src.items():
    print(f"  {label.get(src, src)}: {n}件")
llm = plan.get("llm_detection") or {}
print(f"AI検出　　　: {llm.get('status')} {llm.get('detail', '')}")
print(f"plan保存先　: {PLAN_PATH}")

## 7. render（書き出し）【手動方式】

plan をもとに書き出します。エンコーダは `auto` の場合、NVENC が使える環境では NVENC が選ばれます。
実際に使われたエンコーダと所要時間を表示します。出力は Drive の出力フォルダに保存されます。

In [ ]:
import re, subprocess, sys, time

cmd = [sys.executable, "edit.py", "render",
       "--plan", PLAN_PATH,
       "--thumb", ASSET_FILE,
       "-o", OUTPUT_PATH,
       "--encoder", ENCODER]

t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.time() - t0
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("書き出しに失敗しました")

m = re.search(r"エンコーダ: (\S+)", r.stdout)
print("=" * 50)
print(f"使われたエンコーダ: {m.group(1) if m else '不明（上のログを確認）'}")
print(f"所要時間　　　　　: {elapsed / 60:.1f} 分")
print(f"出力　　　　　　　: {OUTPUT_PATH}")

## 8. 結果確認【手動方式】

出力ファイルの尺とサイズを表示します。ファイルは Drive の出力フォルダにあります。
UIで書き出した場合も、設定セルの OUTPUT_PATH が実際の出力と一致していれば確認に使えます。

In [ ]:
import os, sys
sys.path.insert(0, "/content/zoom-video-auto-editor")
from edit import probe  # ffmpeg の情報から尺を読む（このツールの関数を流用）

info = probe(OUTPUT_PATH)
dur = float(info["format"]["duration"])
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"出力ファイル: {OUTPUT_PATH}")
print(f"尺　　　　　: {int(dur // 60)}:{int(dur % 60):02d}")
print(f"サイズ　　　: {size_mb:.1f} MB")